# Course notebook — Transformers and Foundation Models

Understand the architecture, training, inference, and model families behind modern generative AI.

Use Python 3.10 or later. All inputs are included and the workbook runs without API keys. Each lecture demonstration is a small model of the lecture mechanism. Run it, compare the result with your prediction, and use the section checkpoint to record what you learned.


## Sequence models and attention


### Lecture 01 · Sequence Models Before Transformers

Understand n-grams, recurrent networks, LSTMs, encoder–decoder models, bottlenecks, and why long-range dependencies and serial computation were difficult.

**Predict before running.** Architectures evolve in response to information-flow and optimization bottlenecks. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Train an n-gram model and inspect where local context fails.


In [ ]:
"""Lab 4.1: Sequence Models Before Transformers"""

CHAPTER = "4.1"
print("chapter hook:", CHAPTER)
from collections import Counter
text = "region east failover region west failover"
n = 3
grams = Counter(tuple(text.split()[i:i+n]) for i in range(len(text.split())-n+1))
context = ("region", "east")
candidates = [g[-1] for g in grams if g[:2] == context]
print({"context": context, "next_token_candidates": candidates})
print("---")
print("change one input above, predict output, re-run")


### Lecture 02 · Attention

Build attention from queries, keys, values, similarity scores, normalization, and weighted aggregation.

**Predict before running.** Attention is content-dependent routing of information. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Implement scaled dot-product attention and visualize weights.


In [ ]:
"""Lab 4.2: Attention"""

import math
q = [1.0, 0.0]
keys = [[0.9, 0.1], [0.0, 1.0], [0.9, 0.1]]
def scaled_dot(q, k, scale):
    return sum(a*b for a, b in zip(q, k)) / scale
scale = math.sqrt(len(q))
scores = [scaled_dot(q, k, scale) for k in keys]
m = max(scores)
weights = [math.exp(s-m) for s in scores]
Z = sum(weights)
weights = [w/Z for w in weights]
print("weights:", [round(w, 3) for w in weights])


**Section checkpoint.** Trace one attention query and identify which key contributes most to the weighted result.


## Transformer blocks and training


### Lecture 03 · The Transformer Block

Compose multi-head attention, feed-forward layers, residual paths, normalization, positional information, and masking.

**Predict before running.** Depth repeatedly mixes information and transforms representations. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Assemble one transformer block and test tensor shapes.


In [ ]:
"""Lab 4.3: The Transformer Block"""

seq, dim, heads = 4, 8, 2
assert dim % heads == 0
head_dim = dim // heads
x = [[0.1 * (i+j) for j in range(dim)] for i in range(seq)]
def layer_norm(row):
    mu = sum(row) / len(row)
    var = sum((v-mu)**2 for v in row) / len(row)
    return [(v-mu)/(var+1e-5)**0.5 for v in row]
out = [layer_norm(row) for row in x]
print({"seq": seq, "dim": dim, "head_dim": head_dim, "row0_norm_mean": round(sum(out[0])/len(out[0]), 3)})


### Lecture 04 · Training Foundation Models

Study autoregressive, masked, and sequence-to-sequence objectives; data mixtures; scaling; checkpoints; and mixture-of-experts.

**Predict before running.** Pretraining compresses statistical regularities into parameters; it does not create a fact database. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Estimate compute and data requirements for a tiny language model.


In [ ]:
"""Lab 4.4: Training Foundation Models"""

CHAPTER = "4.4"
print("chapter hook:", CHAPTER)
tokens, params, epochs, batch = 10_000_000, 50_000_000, 1, 32
steps = tokens // (batch * 512)
ratio = tokens / params
print({"train_steps_approx": steps, "tokens_per_param": round(ratio, 2)})
print("note: pretrain learns distributions, not a queryable policy DB")
print("---")
print("change one input above, predict output, re-run")


**Section checkpoint.** Draw the path through attention and the feed-forward block. Mark where training computes a loss.


## Decoding and model selection


### Lecture 05 · Inference and Sampling

Trace logits, softmax, temperature, top-k, top-p, streaming, batching, KV cache, prefix cache, and speculative decoding.

**Predict before running.** Generation is repeated conditional prediction shaped by decoding and system context. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Build a sampling playground and compare decoding strategies.


In [ ]:
"""Lab 4.5: Inference and Sampling"""

import random
logits = [2.0, 1.0, 0.5, 0.1]
def sample_temp(logits, temp=1.0):
    scaled = [l/temp for l in logits]
    m = max(scaled)
    ex = [math.exp(l-m) for l in scaled]
    s = sum(ex)
    probs = [e/s for e in ex]
    r = random.random()
    c = 0
    for i, p in enumerate(probs):
        c += p
        if r <= c:
            return i, probs
    return len(probs)-1, probs
import math
idx, probs = sample_temp(logits, temp=0.8)
print({"sampled_index": idx, "probs": [round(p, 3) for p in probs]})


### Lecture 06 · Model Families and Selection

Compare base, instruction, reasoning, code, embedding, reranking, reward, safety, speech, vision, and diffusion models.

**Predict before running.** Select models as replaceable components against requirements, not by reputation. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Benchmark candidate models on a task-specific dataset.


In [ ]:
"""Lab 4.6: Model Families and Selection"""

tasks = {"lookup": 0.95, "cite": 0.88, "route": 0.91}
models = {"small-instruct": 0.01, "large-reason": 0.08}
def route(task, risk):
    if risk == "low" and tasks[task] > 0.9:
        return "small-instruct"
    return "large-reason"
for risk in ("low", "high"):
    print({"risk": risk, "model": route("lookup", risk), "cost_per_1k": models[route("lookup", risk)]})


**Section checkpoint.** Compare decoding distributions at two temperatures. Pick a model selection metric for repeatability.


## Course workshop

Keep logits fixed, vary temperature, and compare the probability of the leading token. Explain why selection still needs a task-level evaluation.

Run the reference implementation, then run the checks below it.


In [ ]:
"""Scaled dot-product attention and temperature sampling without dependencies."""
from math import exp, sqrt
from random import Random


def softmax(values, temperature=1.0):
    scaled = [v / temperature for v in values]; peak = max(scaled)
    weights = [exp(v - peak) for v in scaled]; total = sum(weights)
    return [w / total for w in weights]


def attention(query, keys, values):
    scores = [sum(q * k for q, k in zip(query, key)) / sqrt(len(query)) for key in keys]
    weights = softmax(scores)
    output = [sum(weight * value[i] for weight, value in zip(weights, values)) for i in range(len(values[0]))]
    return weights, output


weights, output = attention([1, 0], [[1, 0], [0, 1]], [[10, 0], [0, 10]])
print("attention_weights=", [round(x, 3) for x in weights], "output=", [round(x, 3) for x in output])
for temperature in (0.3, 1.0, 2.0):
    probabilities = softmax([3.0, 2.0, 1.0], temperature)
    print(f"temperature={temperature}: {[round(x, 3) for x in probabilities]}")


### Verify and extend

Run the assertions. Then change one assumption and add a check for the predicted behavior.


In [ ]:
cold = softmax([2.0, 1.0, 0.0], temperature=0.5)
warm = softmax([2.0, 1.0, 0.0], temperature=2.0)
assert abs(sum(cold) - 1.0) < 1e-9
assert cold[0] > warm[0]
print('Workshop checks passed')


## Course project

Implement a tiny transformer and create a vendor-neutral model selection report.

Keep your baseline, results, and one failure case so someone else can review the decision you made.
